# 81 — Responder idea lab (cheap offline Gemini A/B over Blind-A)

Screens responder prompt strategies CHEAPLY before spending Blind-A submissions.
Why this exists: the leaderboard LLM judge is non-deterministic (~±0.25 on 80 rows),
so single submissions cannot tell a real responder gain from noise. Here we use the
SAME Gemini judge offline, on a SMALL sample, with REPEATS, to estimate mean ± spread.

How to use:
- Run cell 1 (setup) and cell 2 (engine) once.
- Each idea below is ONE self-contained cell that calls `evaluate(label, respond_fn)`.
- Run `baseline` first, then any ideas, then the SUMMARY cell.
- DECISION RULE: only trust a variant whose mean total beats baseline by MORE than the
  ±std printed (that std is your judge-noise band). Re-run a cell to see run-to-run wobble.

Test-time compute: generation runs on the cheap LITE model so we can afford MANY refinement
passes per response (iterative self-critique). This is "test-time compute" (more inference
per example) — the practical cousin of test-time training (which would update weights per
example; not available via the API).

COST: we pay per Gemini call. Keep SAMPLE/REPEATS small. Rough calls per simple variant
= SAMPLE*REPEATS gen + SAMPLE*REPEATS judge; iter-refine multiplies gen by (1+REFINE_ITERS);
best-of-N multiplies gen by N. Lite is ~cheapest, so the refine loops stay affordable.

CAVEAT (read the summary): best-of-N / adaptive-refine SELECT using the same judge that
scores them -> their offline numbers are optimistically biased vs methods that don't peek.
Some of that transfers to the leaderboard (same judge family), but discount it.

In [ ]:
# 1) Setup — clone branch + Gemini key + Drive + light deps (no GPU).
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'
from google.colab import userdata, drive
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # add this Colab secret first
drive.mount('/content/drive', force_remount=False)
BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -q -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!pip install -q -U google-generativeai 'datasets' 'pandas<3.0'
print('setup done | GEMINI key present:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# 2) Engine — knobs, data, judge (both axes), and the evaluate() harness.
import os, re, json, time, math, statistics as st, sys
sys.path.insert(0, '/content/recsys2026/scripts')
import gemini_responder as gr                 # reuse the tested building blocks
import google.generativeai as genai
genai.configure(api_key=os.environ['GEMINI_API_KEY'])

# ---- KNOBS (keep small — we pay per call) ----
SAMPLE   = 8                          # Blind-A sessions to test on
REPEATS  = 3                          # repeats per session -> estimates judge+gen noise
TOP_N    = 3                          # tracks shown to the responder
REFINE_ITERS = 4                      # test-time compute: self-refine passes (lite is cheap)
GEN_MODEL   = 'gemini-2.5-flash-lite' # cheap generator -> afford many refine passes
JUDGE_MODEL = 'gemini-2.5-flash'      # judge kept stronger for reliable selection/scoring
SLEEP    = 0.1
CANDIDATES_SRC = '/content/drive/MyDrive/recsys2026/194-union-sasrec-lgbm-cleanfull-v5kto-blindA.json'

gen_model   = genai.GenerativeModel(GEN_MODEL)
judge_model = genai.GenerativeModel(JUDGE_MODEL)

from datasets import load_dataset
ds = load_dataset('talkpl-ai/TalkPlayData-Challenge-Blind-A', split='test')
sess_by_id = {s['session_id']: s for s in ds}
item_db_meta = gr._load_item_meta()
preds = json.load(open(CANDIDATES_SRC))[:SAMPLE]
print(f'loaded {len(preds)} candidate rows (SAMPLE={SAMPLE}), models gen={GEN_MODEL} judge={JUDGE_MODEL}')

# ---- generation + judging helpers ----
def gen(prompt, temp=0.7):
    return (gr._call_model(gen_model, prompt, gen_config={'temperature': temp}) or '').strip()

def judge_axes(ctx, tracks, reply):
    """Return (personalization, explanation_quality) at judge temp 0, or None."""
    raw = gr._call_model(judge_model, gr.build_judge_prompt(ctx, tracks, reply),
                         gen_config={'temperature': 0.0})
    if not raw: return None
    m = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if not m: return None
    try:
        d = json.loads(m.group(0))
        return float(d['personalization']), float(d['explanation_quality'])
    except Exception:
        return None

# ---- shared prompt builders (reuse gemini_responder; togglable for ablations) ----
def base_prompt(sess, p, ctx, tracks):
    goal = ((sess.get('conversation_goal') or {}).get('listener_goal') or '').strip()
    uc   = gr.render_user_context(sess.get('user_profile'))
    gp   = gr.goal_progress_tally(sess.get('goal_progress_assessments'), p['turn_number'])
    return gr.build_prompt(ctx, tracks, goal, user_context=uc, goal_progress=gp)

def strip_thought(ctx):  return re.sub(r' \| why: [^\]]*\]', ']', ctx)   # ablate prior-thought
def strip_era(tracks):   return re.sub(r' — era \d{4}s', '', tracks)        # ablate era

def refine_once(ctx, tracks, draft, temp=0.4):
    """One test-time-compute pass: critique the DRAFT against the two judge axes and rewrite it."""
    crit = ("Revise the DRAFT music-recommendation reply so it (1) ties to something the user ACTUALLY "
            "said in the conversation, and (2) cites one concrete attribute of the recommended track. "
            "Keep it 2-3 sentences, warm, no preamble. Output ONLY the reply text.\n\n"
            f"=== CONVERSATION ===\n{ctx}\n\n=== TRACKS ===\n{tracks}\n\nDRAFT:\n{draft}\n\nImproved reply:")
    return gen(crit, temp=temp) or draft

RESULTS = {}
def evaluate(label, respond_fn):
    """respond_fn(sess, p, ctx, tracks) -> reply. Scores SAMPLE x REPEATS turns.
    Stores per-session means (repeats averaged) so the SUMMARY can do a PAIRED
    delta vs baseline (cancels session-difficulty -> resolves small real gains).
    Prints mean ± SEM (SEM = std/sqrt(n) = uncertainty of the MEAN, not the raw spread)."""
    rows, by_sess = [], {}
    for p in preds:
        sess = sess_by_id.get(p['session_id'])
        if not sess: continue
        tn = p['turn_number']
        ctx    = gr.render_context(sess['conversations'], item_db_meta, tn)
        tracks = gr.format_tracks(p.get('predicted_track_ids'), item_db_meta, n=TOP_N)
        for _ in range(REPEATS):
            reply = respond_fn(sess, p, ctx, tracks)
            sc = judge_axes(ctx, tracks, reply) if reply else None
            if sc:
                rows.append(sc)
                by_sess.setdefault(p['session_id'], []).append(sc[0] + sc[1])
            time.sleep(SLEEP)
    if not rows:
        print(f'{label}: NO SCORES (check API/key)'); return
    tot = [a + b for a, b in rows]
    std = st.pstdev(tot) if len(tot) > 1 else 0.0
    res = dict(n=len(rows),
               pers=st.mean(r[0] for r in rows),
               expl=st.mean(r[1] for r in rows),
               total=st.mean(tot),
               std=std,
               sem=std / math.sqrt(len(tot)) if tot else 0.0,
               by_session={sid: st.mean(v) for sid, v in by_sess.items()})  # one score / session
    RESULTS[label] = res
    print(f'{label:26} n={res["n"]:3d}  pers={res["pers"]:.2f}  expl={res["expl"]:.2f}  '
          f'total={res["total"]:.2f} ±{res["std"]:.2f}  (SEM {res["sem"]:.2f})')
    return res

print('engine ready. Run the baseline cell, then any idea cells, then SUMMARY.')

In [ ]:
# 3) BASELINE — current consolidated responder (all enrichments on, single-shot).
def respond_baseline(sess, p, ctx, tracks):
    return gen(base_prompt(sess, p, ctx, tracks))
evaluate('baseline (current)', respond_baseline)

In [ ]:
# 4) IDEA 1 — SELF-REFINE (generate -> 1 critique/revise pass).
# Robust to judge noise (improves the single output; doesn't rely on a noisy judge to pick).
def respond_self_refine(sess, p, ctx, tracks):
    return refine_once(ctx, tracks, gen(base_prompt(sess, p, ctx, tracks)))
evaluate('self-refine (1 pass)', respond_self_refine)

In [ ]:
# 4b) IDEA 1b — ITERATIVE REFINE (test-time compute): REFINE_ITERS critique/revise passes.
# "More thinking iterations" — cheap on the lite generator. Tune REFINE_ITERS in cell 2.
def respond_iter_refine(sess, p, ctx, tracks):
    r = gen(base_prompt(sess, p, ctx, tracks))
    for _ in range(REFINE_ITERS):
        r = refine_once(ctx, tracks, r)
    return r
evaluate(f'iter-refine x{REFINE_ITERS}', respond_iter_refine)

In [ ]:
# 4c) IDEA 1c — ADAPTIVE REFINE: refine up to REFINE_ITERS, keep the self-judged BEST draft,
# stop early after 2 passes with no gain (spend compute only while it helps).
# NOTE: selects via the same judge that scores -> offline number is optimistically biased.
def respond_adaptive(sess, p, ctx, tracks):
    r = gen(base_prompt(sess, p, ctx, tracks))
    best, best_sum, stale = r, sum(judge_axes(ctx, tracks, r) or (0, 0)), 0
    for _ in range(REFINE_ITERS):
        r = refine_once(ctx, tracks, r)
        s = judge_axes(ctx, tracks, r)
        if s and sum(s) > best_sum:
            best, best_sum, stale = r, sum(s), 0
        else:
            stale += 1
            if stale >= 2:
                break
    return best
evaluate(f'adaptive-refine (<={REFINE_ITERS})', respond_adaptive)

In [ ]:
# 5) IDEA 2 — RUBRIC-ALIGNED BEST-OF-N (N candidates, judge each on the 2 axes, pick best).
N_BEST = 5
def respond_best_of_n(sess, p, ctx, tracks):
    return gr.generate_best_of_n(
        gen_model, judge_model, base_prompt(sess, p, ctx, tracks), ctx, tracks,
        fallback=(p.get('predicted_response') or ''), n=N_BEST,
        temperatures=[0.5, 0.7, 0.9, 1.0, 1.1])
evaluate(f'best-of-{N_BEST} (rubric)', respond_best_of_n)

In [ ]:
# 6) IDEA 3 — FORCED DUAL-CITATION (lean: name 1 user phrase + 1 track attribute, connect them).
# Leaner than the rejected structured_personality CoT (which over-name-dropped at top_n=3).
DUAL = ("You are a music recommender replying mid-conversation. Silently identify (a) the most important "
        "thing the user ACTUALLY said they want, and (b) one concrete attribute of the recommended track. "
        "Then write a warm 2-3 sentence reply that explicitly connects (a) to (b). Lead with the pick. "
        "No preamble, no lists, no quotation marks. Output ONLY the reply.")
def respond_dual(sess, p, ctx, tracks):
    goal = ((sess.get('conversation_goal') or {}).get('listener_goal') or '').strip()
    prompt = (DUAL + "\n\n=== CONVERSATION ===\n" + ctx
              + (("\n\nListener goal: " + goal) if goal else "")
              + "\n\n=== RECOMMENDED TRACK(S) ===\n" + tracks + "\n\nReply:")
    return gen(prompt)
evaluate('forced-dual-citation', respond_dual)

In [ ]:
# 7) IDEA 4 — ABLATIONS of the enrichments we added (free: just toggle pieces of the prompt).
def _abl(use_thought=True, use_era=True, use_culture=True, use_goalprog=True):
    def f(sess, p, ctx, tracks):
        c = ctx if use_thought else strip_thought(ctx)
        t = tracks if use_era else strip_era(tracks)
        goal = ((sess.get('conversation_goal') or {}).get('listener_goal') or '').strip()
        uc = gr.render_user_context(sess.get('user_profile')) if use_culture else ''
        gp = gr.goal_progress_tally(sess.get('goal_progress_assessments'), p['turn_number']) if use_goalprog else ''
        return gen(gr.build_prompt(c, t, goal, user_context=uc, goal_progress=gp))
    return f
evaluate('abl: no-culture',  _abl(use_culture=False))
evaluate('abl: no-goalprog', _abl(use_goalprog=False))
evaluate('abl: no-thought',  _abl(use_thought=False))
evaluate('abl: minimal',     _abl(False, False, False, False))

In [ ]:
# 8) IDEA 5 — LENGTH CALIBRATION (does the judge prefer terser / longer replies?).
def respond_len(n):
    def f(sess, p, ctx, tracks):
        prompt = base_prompt(sess, p, ctx, tracks).replace('2-3 short sentences', f'exactly {n} sentences')
        return gen(prompt)
    return f
for n in (2, 3, 4):
    evaluate(f'length-{n}-sentences', respond_len(n))

In [ ]:
# 9) IDEA 6 — LIGHT INTENT-MIRRORING (re-test: the "don't echo" rule was set on a noisy signal).
MIRROR = ("\n\nBegin by briefly restating, in your own words (not verbatim), the specific thing the user "
          "asked for, then make the recommendation.")
def respond_mirror(sess, p, ctx, tracks):
    return gen(base_prompt(sess, p, ctx, tracks) + MIRROR)
evaluate('intent-mirroring', respond_mirror)

In [ ]:
# 10) SUMMARY — mean ± SEM, plus PAIRED delta vs baseline (same sessions, repeats averaged).
# The paired test cancels session-difficulty, so it resolves much smaller real gains than
# comparing two means each with their big raw ±std. A variant is a WIN only if its paired
# delta clears ~2x its SE (and is > 0); otherwise it is indistinguishable from baseline.
base = RESULTS.get('baseline (current)')
print(f'{"variant":26} {"total":>6} {"SEM":>5} | {"Δ vs base":>9} {"±SE":>5} {"verdict":>8}')
print('-' * 70)
for k, v in sorted(RESULTS.items(), key=lambda kv: -kv[1]['total']):
    line = f'{k:26} {v["total"]:6.2f} {v["sem"]:5.2f} | '
    if k == 'baseline (current)':
        line += f'{"(baseline)":>9}'
    elif base:
        sids = [s for s in v['by_session'] if s in base['by_session']]
        deltas = [v['by_session'][s] - base['by_session'][s] for s in sids]
        if len(deltas) >= 2:
            md = st.mean(deltas)
            se = st.pstdev(deltas) / math.sqrt(len(deltas))
            verdict = 'WIN' if md - 2 * se > 0 else ('loss' if md + 2 * se < 0 else 'noise')
            line += f'{md:+9.2f} {se:5.2f} {verdict:>8}'
        else:
            line += f'{"n/a":>9}'
    print(line)
print('\nPAIRED across the same {} sessions (repeats averaged per session).'.format(
      len(base['by_session']) if base else 0))
print('WIN = paired Δ > ~2x SE (>0). "noise" = cannot distinguish from baseline at this SAMPLE/REPEATS.')
print('Borderline/close to the bar? bump SAMPLE (more sessions) — that shrinks SE fastest.')
print('DISCOUNT best-of-N / adaptive-refine: they select via the same judge that scores -> inflated.')

In [ ]:
# 11) DE-BIASED N-SWEEP — where does best-of-N's REAL gain plateau?
# Selection and scoring use DIFFERENT judges, so the number is NOT inflated by picking-and-
# scoring with the same model (the winner's curse). N=1 (independently scored) is the baseline;
# paired delta vs N=1 across the same sessions. Find the N where total stops rising.
N_GRID = [1, 3, 5, 6, 8]
SELECT_MODEL = genai.GenerativeModel('gemini-2.5-flash')        # picks the best of N
SCORE_MODEL  = genai.GenerativeModel('gemini-2.5-flash-lite')   # INDEPENDENT scorer (de-bias)

def _score_indep(ctx, tracks, reply):
    raw = gr._call_model(SCORE_MODEL, gr.build_judge_prompt(ctx, tracks, reply), gen_config={'temperature': 0.0})
    if not raw: return None
    m = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if not m: return None
    try:
        d = json.loads(m.group(0)); return float(d['personalization']) + float(d['explanation_quality'])
    except Exception:
        return None

NSWEEP = {}
for N in N_GRID:
    by_sess = {}
    for p in preds:
        sess = sess_by_id.get(p['session_id'])
        if not sess: continue
        tn = p['turn_number']
        ctx = gr.render_context(sess['conversations'], item_db_meta, tn)
        tracks = gr.format_tracks(p.get('predicted_track_ids'), item_db_meta, n=TOP_N)
        for _ in range(REPEATS):
            if N == 1:
                reply = gen(base_prompt(sess, p, ctx, tracks))
            else:
                reply = gr.generate_best_of_n(gen_model, SELECT_MODEL, base_prompt(sess, p, ctx, tracks),
                                              ctx, tracks, fallback=(p.get('predicted_response') or ''),
                                              n=N, temperatures=[0.5, 0.7, 0.9, 1.0, 1.1])
            s = _score_indep(ctx, tracks, reply)
            if s is not None:
                by_sess.setdefault(p['session_id'], []).append(s)
            time.sleep(SLEEP)
    NSWEEP[N] = {sid: st.mean(v) for sid, v in by_sess.items()}

base = NSWEEP.get(1, {})
print(f'{"N":>3} {"total":>6} {"SEM":>5} | {"d vs N=1":>9} {"+/-SE":>6} {"verdict":>8}')
print('-' * 50)
for N in N_GRID:
    vals = list(NSWEEP[N].values())
    if not vals:
        print(f'{N:>3}  (no scores)'); continue
    mean = st.mean(vals); sem = st.pstdev(vals) / math.sqrt(len(vals)) if len(vals) > 1 else 0.0
    line = f'{N:>3} {mean:6.2f} {sem:5.2f} | '
    if N == 1:
        line += f'{"(base)":>9}'
    else:
        sids = [s for s in NSWEEP[N] if s in base]
        d = [NSWEEP[N][s] - base[s] for s in sids]
        if len(d) >= 2:
            md = st.mean(d); se = st.pstdev(d) / math.sqrt(len(d))
            verdict = 'WIN' if md - 2 * se > 0 else ('loss' if md + 2 * se < 0 else 'noise')
            line += f'{md:+9.2f} {se:6.2f} {verdict:>8}'
        else:
            line += f'{"n/a":>9}'
    print(line)
print('\nINDEPENDENT scorer (select=flash, score=flash-lite) -> de-biased estimate.')
print('Pick the smallest N at the plateau: beyond it you pay ~Nx more for no real gain,')
print('and selecting harder on judge noise can start to HURT the real leaderboard.')